# Cassava disease classifier

Trains the model AgroMet ships on the phone. Run on Kaggle with the
**Cassava Leaf Disease Classification** dataset attached and a **T4 GPU**
accelerator selected.

Output: `cassava.keras`, which `ml/scripts/export_tflite.py` converts to the
`.tflite` asset the app bundles.

## Why this dataset

Roughly 21,000 photographs taken in the field on ordinary phones, in Uganda.
Cluttered backgrounds, uneven light, several leaves in frame, some shots
blurred. That is what an AgroMet user will actually submit.

The obvious alternative, PlantVillage, is 54,000 images of single detached
leaves on a uniform grey background under studio light. Models trained on it
report accuracies close to 99% and then fall over on real photographs.
Notebook `02_train_plantvillage.ipynb` trains that version deliberately so the
size of the gap can be measured rather than assumed.

## Why MobileNetV3-Small

The deployment constraint picks the architecture. Inference runs on a mid-range
Android phone, offline, so the model has to be small enough to bundle and fast
enough to feel instant. A larger backbone would score better and could not ship.

In [ ]:
import json, os, pathlib, random
import numpy as np
import pandas as pd
import tensorflow as tf
import keras

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMAGE_SIZE = 224
BATCH_SIZE = 32
# Kaggle's own class order. This must match CLASS_IDS in ml/scripts/model_def.py:
# the index the model outputs only means a disease because both sides agree.
CLASS_IDS = ["cbb", "cbsd", "cgm", "cmd", "healthy"]

DATA_DIR = pathlib.Path("/kaggle/input/cassava-leaf-disease-classification")
OUT_DIR = pathlib.Path("/kaggle/working")

print("GPU:", tf.config.list_physical_devices("GPU"))

## Load the labels

`train.csv` maps each filename to a label index 0-4. The competition's
`label_num_to_disease_map.json` confirms the order.

In [ ]:
labels = pd.read_csv(DATA_DIR / "train.csv")
with open(DATA_DIR / "label_num_to_disease_map.json") as handle:
    label_names = json.load(handle)

print(labels.shape)
print(label_names)

# Heavily imbalanced: CMD alone is over 60% of the set. Left alone the model
# learns to answer "CMD" and is right most of the time while being useless.
counts = labels["label"].value_counts().sort_index()
print(counts)
print((counts / counts.sum()).round(3))

## Split

Stratified, so every class keeps its proportion in both halves. A plain random
split can leave the rarest class (CBB, about 5%) with too few validation
examples for its recall to mean anything.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    labels, test_size=0.15, stratify=labels["label"], random_state=SEED
)
print(len(train_df), len(val_df))

## Input pipeline

Pixels stay at 0-255. MobileNetV3 in Keras carries its own rescaling layer, so
normalising here would halve the input range and cost accuracy silently. The
app's `preprocess.ts` leaves them at 0-255 for the same reason, and
`labels.json` records the contract next to the model.

In [ ]:
IMAGE_DIR = DATA_DIR / "train_images"

def make_dataset(frame, *, training):
    paths = [str(IMAGE_DIR / name) for name in frame["image_id"]]
    ds = tf.data.Dataset.from_tensor_slices((paths, frame["label"].values))

    def load(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
        return image, label  # float32 in 0-255, deliberately not rescaled

    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2048, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)

## Augmentation

Each transform stands for a real difference between a training photograph and
the one a farmer will take:

| Transform | The field condition it stands for |
|---|---|
| Horizontal and vertical flip | A leaf has no canonical orientation |
| Rotation (±20%) | The phone is held at whatever angle is convenient |
| Brightness and contrast | Overcast morning against midday sun |
| Random zoom | How close the farmer chose to stand |

Deliberately absent: hue and heavy saturation shifts. Colour *is* the symptom
for mosaic and vein chlorosis, so distorting it trains the model to ignore the
evidence it needs.

In [ ]:
augment = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    keras.layers.RandomRotation(0.2, seed=SEED),
    keras.layers.RandomZoom(0.2, seed=SEED),
    keras.layers.RandomBrightness(0.2, value_range=(0, 255), seed=SEED),
    keras.layers.RandomContrast(0.2, seed=SEED),
], name="augment")

# Applied in the input pipeline, NOT as a layer inside the model.
#
# Keras augmentation layers are inference-time no-ops, so putting them in the
# model looks harmless. They are still exported into the TFLite graph, and the
# shipped model then stops being the network model_def.py describes. Keeping
# them out here is what lets that file stay the single definition of what gets
# converted.
train_ds = train_ds.map(
    lambda images, labels: (augment(images, training=True), labels),
    num_parallel_calls=tf.data.AUTOTUNE,
)

In [ ]:
def build_model(num_classes=len(CLASS_IDS), weights="imagenet"):
    """Mirrors ml/scripts/model_def.py. Kept inline so the notebook runs on
    Kaggle, where this repository is not checked out."""
    backbone = keras.applications.MobileNetV3Small(
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
        include_top=False,
        weights=weights,
        include_preprocessing=True,
    )
    backbone.trainable = False

    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), dtype="float32", name="image")
    x = backbone(inputs, training=False)
    x = keras.layers.GlobalAveragePooling2D(name="pool")(x)
    x = keras.layers.Dropout(0.2, name="dropout")(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax", name="probabilities")(x)
    return keras.Model(inputs, outputs, name="cassava_mobilenetv3s"), backbone

model, backbone = build_model()
model.summary()

## Class weights

CMD is over 60% of the data and CBB around 5%. Weighting the loss by inverse
frequency makes a missed CBB cost more than a missed CMD, which is also the
right trade in the field: the majority class is the one a farmer could most
easily guess without an app.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(
    "balanced", classes=np.arange(len(CLASS_IDS)), y=train_df["label"].values
)
class_weight = dict(enumerate(weights))
print({CLASS_IDS[i]: round(w, 3) for i, w in class_weight.items()})

## Stage 1: train the head

The backbone stays frozen. Its ImageNet features are already good at leaves and
edges, and letting a randomly initialised head send large gradients into them
would destroy that before it learns anything.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

history_head = model.fit(
    train_ds, validation_data=val_ds, epochs=10,
    class_weight=class_weight, callbacks=callbacks,
)

## Stage 2: fine-tune the top of the backbone

Unfreeze the last blocks at a much lower learning rate. The early layers stay
frozen: generic edge and texture filters transfer fine and retraining them on
21,000 images mostly overfits.

1e-5 rather than 1e-3. At the higher rate fine-tuning wipes out the pretrained
features and validation accuracy drops below where stage 1 left it.

In [ ]:
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_ft = model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    class_weight=class_weight, callbacks=callbacks,
)

## Per-class results

Overall accuracy is the wrong headline on an imbalanced set: answering "CMD"
every time scores over 60% while being worthless. Per-class recall is what says
whether the model can find the rare diseases, and the confusion matrix is what
the report needs.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = np.concatenate([y.numpy() for _, y in val_ds])
y_pred = model.predict(val_ds).argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=CLASS_IDS, digits=3))
print(confusion_matrix(y_true, y_pred))

In [ ]:
model.save(OUT_DIR / "cassava.keras")
print("saved", OUT_DIR / "cassava.keras")

# Download this, put it in ml/models/, then:
#   python ml/scripts/export_tflite.py --model ml/models/cassava.keras